In [1]:
import pandas as pd
import glob

In [21]:
kaggle_files = glob.glob("./*.csv")
kaggle_dfs = [pd.read_csv(file) for file in kaggle_files]
df_kaggle = pd.concat(kaggle_dfs, ignore_index=True)

In [10]:
df_kaggle.tail()

,latitude,longitude,postal_code,address,closest_mrt,closest_mrt_dist,cbd_dist,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,years_remaining,remaining_lease
896641,1.338745,103.847253,311099.0,99B LOR 2 TOA PAYOH,Braddell MRT Station,197.208781,6180.426023,2014-09,TOA PAYOH,EXECUTIVE,99B,LOR 2 TOA PAYOH,10 TO 12,145.0,Apartment,1993,850000.0,NaN,78,NaN
896642,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2012-03,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,11 TO 15,149.0,Apartment,1993,862000.0,NaN,80,NaN
896643,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2012-06,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,04 TO 06,148.0,Apartment,1993,820000.0,NaN,80,NaN
896644,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2013-10,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,10 TO 12,148.0,Apartment,1993,905000.0,NaN,79,NaN
896645,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2013-12,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,07 TO 09,145.0,Apartment,1993,845000.0,NaN,79,NaN


In [22]:
df_kaggle.dtypes

latitude               float64
longitude              float64
postal_code            float64
address                 object
closest_mrt             object
closest_mrt_dist       float64
cbd_dist               float64
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
resale_price           float64
year                   float64
years_remaining          int64
remaining_lease         object
dtype: object

In [ ]:
df_kaggle["postal_code"] = pd.to_numeric(df_kaggle["postal_code"], errors="coerce").astype("Int64")
df_kaggle["resale_price"] = pd.to_numeric(df_kaggle["resale_price"], errors="coerce").astype("int64")
df_kaggle["floor_area_sqm"] = pd.to_numeric(df_kaggle["floor_area_sqm"], errors="coerce").astype("int64")
df_kaggle["resale_price"] = pd.to_numeric(df_kaggle["resale_price"], errors="coerce").astype("int64")

# sort by month
df_kaggle = df_kaggle.sort_values(by=["month", "street_name"])
df_kaggle["month"] = pd.to_datetime(df_kaggle["month"], format="%Y-%m")

df_kaggle = df_kaggle.drop(columns=["year", "remaining_lease"], errors="ignore")

# year & month as separate columns
df_kaggle["year"] = df_kaggle["month"].dt.year
df_kaggle["month"] = df_kaggle["month"].dt.month

In [24]:
df_kaggle.head()

,latitude,longitude,postal_code,address,closest_mrt,closest_mrt_dist,cbd_dist,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,years_remaining,year
3000,1.370262,103.839468,560101,101 ANG MO KIO AVE 3,Mayflower MRT Station,349.028529,9738.375384,1,ANG MO KIO,5 ROOM,101,ANG MO KIO AVE 3,07 TO 09,117,STANDARD,1978,118000,87,1990
5540,1.371017,103.839305,560103,103 ANG MO KIO AVE 3,Mayflower MRT Station,308.580046,9823.548103,1,ANG MO KIO,4 ROOM,103,ANG MO KIO AVE 3,07 TO 09,90,NEW GENERATION,1978,64500,87,1990
7662,1.372313,103.837601,560105,105 ANG MO KIO AVE 4,Mayflower MRT Station,148.527965,9992.785531,1,ANG MO KIO,4 ROOM,105,ANG MO KIO AVE 4,04 TO 06,92,NEW GENERATION,1978,72500,87,1990
14473,1.370459,103.837419,560110,110 ANG MO KIO AVE 4,Mayflower MRT Station,145.945087,9793.411075,1,ANG MO KIO,3 ROOM,110,ANG MO KIO AVE 4,10 TO 12,67,NEW GENERATION,1978,34000,87,1990
22806,1.373717,103.835610,560117,117 ANG MO KIO AVE 4,Mayflower MRT Station,271.039657,10181.864756,1,ANG MO KIO,3 ROOM,117,ANG MO KIO AVE 4,04 TO 06,74,NEW GENERATION,1978,37000,87,1990


In [ ]:
# split 'storey_range' into lower & upper bounds
def split_storey_range(storey_range):
    try:
        lower, upper = storey_range.split(" TO ")
        return int(lower), int(upper)
    except ValueError:
        return None, None

df_kaggle[["storey_lower", "storey_upper"]] = df_kaggle["storey_range"].apply(
    lambda x: pd.Series(split_storey_range(x))
)

df_kaggle = df_kaggle.drop(columns=["storey_range"], errors="ignore")

In [29]:
df_kaggle.columns

Index(['latitude', 'longitude', 'postal_code', 'address', 'closest_mrt',
       'closest_mrt_dist', 'cbd_dist', 'month', 'town', 'flat_type', 'block',
       'street_name', 'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'resale_price', 'years_remaining', 'year', 'storey_lower',
       'storey_upper'],
      dtype='object')

In [31]:
column_order = [
    "year", "month",
    "latitude", "longitude", "postal_code",
    "town", "block", "street_name", 
    "flat_type", "storey_lower", "storey_upper", "floor_area_sqm", "flat_model", "lease_commence_date", "years_remaining",
    "closest_mrt", "closest_mrt_dist", "cbd_dist",
    "resale_price"
]

df_kaggle = df_kaggle[column_order]

In [32]:
df_kaggle

,year,month,latitude,longitude,postal_code,town,block,street_name,flat_type,storey_lower,storey_upper,floor_area_sqm,flat_model,lease_commence_date,years_remaining,closest_mrt,closest_mrt_dist,cbd_dist,resale_price
3000,1990,1,1.370262,103.839468,560101,ANG MO KIO,101,ANG MO KIO AVE 3,5 ROOM,7,9,117,STANDARD,1978,87,Mayflower MRT Station,349.028529,9738.375384,118000
5540,1990,1,1.371017,103.839305,560103,ANG MO KIO,103,ANG MO KIO AVE 3,4 ROOM,7,9,90,NEW GENERATION,1978,87,Mayflower MRT Station,308.580046,9823.548103,64500
7662,1990,1,1.372313,103.837601,560105,ANG MO KIO,105,ANG MO KIO AVE 4,4 ROOM,4,6,92,NEW GENERATION,1978,87,Mayflower MRT Station,148.527965,9992.785531,72500
14473,1990,1,1.370459,103.837419,560110,ANG MO KIO,110,ANG MO KIO AVE 4,3 ROOM,10,12,67,NEW GENERATION,1978,87,Mayflower MRT Station,145.945087,9793.411075,34000
22806,1990,1,1.373717,103.835610,560117,ANG MO KIO,117,ANG MO KIO AVE 4,3 ROOM,4,6,74,NEW GENERATION,1978,87,Mayflower MRT Station,271.039657,10181.864756,37000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
833499,2023,4,1.417389,103.832369,762813,YISHUN,813B,YISHUN RING RD,3 ROOM,7,9,67,Model A,2018,94,Khatib MRT Station,68.040054,15008.676091,450000
834546,2023,4,1.415191,103.832902,760828,YISHUN,828,YISHUN ST 81,EXECUTIVE,10,12,142,Apartment,1988,64,Khatib MRT Station,242.585206,14759.708167,865000
835248,2023,4,1.415715,103.833410,760840,YISHUN,840,YISHUN ST 81,3 ROOM,4,6,73,Model A,1988,64,Khatib MRT Station,190.560783,14809.421378,412000
837710,2023,4,1.413545,103.836971,760872,YISHUN,872,YISHUN ST 81,5 ROOM,1,3,127,Improved,1988,64,Khatib MRT Station,614.307170,14522.828800,640000


In [33]:
df_kaggle.to_csv("cleaned_hdb_resale_data.csv", index=False)

## (A) Town Level

In [40]:
df_town_level_year_flat_type = df_kaggle.groupby(["year", "town", "flat_type"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_remaining_lease=("years_remaining", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_town_level_year_flat_type

,year,town,flat_type,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_remaining_lease,avg_mrt_distance,avg_cbd_distance
0,1990,ANG MO KIO,1 ROOM,7.770833e+03,8000.0,24,31.000000,86.000000,NaN,NaN
1,1990,ANG MO KIO,2 ROOM,2.510833e+04,23000.0,12,45.000000,95.000000,357.423622,10257.700017
2,1990,ANG MO KIO,3 ROOM,4.644389e+04,47000.0,1096,71.499088,88.287409,639.520721,9735.328305
3,1990,ANG MO KIO,4 ROOM,7.706832e+04,75000.0,368,92.942935,88.103261,602.672782,9739.138430
4,1990,ANG MO KIO,5 ROOM,1.307095e+05,130000.0,128,120.351562,88.585938,694.200289,9743.971289
...,...,...,...,...,...,...,...,...,...,...
3834,2023,YISHUN,3 ROOM,3.826468e+05,378000.0,153,68.764706,70.163399,774.735620,16042.123346
3835,2023,YISHUN,4 ROOM,4.843981e+05,479500.0,252,93.690476,74.039683,956.959060,15785.664166
3836,2023,YISHUN,5 ROOM,6.371329e+05,635000.0,95,117.400000,79.284211,940.729756,15740.229401
3837,2023,YISHUN,EXECUTIVE,8.194667e+05,800000.0,15,148.800000,64.066667,883.643875,15867.634211


In [ ]:
df_town_level_year_flat_type.to_csv("resale_data_town_year_flat_type.csv", index=False)

In [42]:
df_town_level_year = df_kaggle.groupby(["year", "town"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_remaining_lease=("years_remaining", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_town_level_year

,year,town,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_remaining_lease,avg_mrt_distance,avg_cbd_distance
0,1990,ANG MO KIO,59264.292383,47200.0,1628,79.394963,88.285012,633.166924,9740.990746
1,1990,BEDOK,70226.199678,57000.0,1242,86.627214,88.518519,808.656018,10189.943945
2,1990,BISHAN,92737.735849,65500.0,106,89.047170,91.358491,413.762159,7820.168766
3,1990,BUKIT BATOK,87756.431611,71400.0,658,98.648936,93.653495,590.111920,13609.767856
4,1990,BUKIT MERAH,65934.881690,50000.0,710,73.200000,84.976056,695.492139,3442.449856
...,...,...,...,...,...,...,...,...,...
867,2023,SERANGOON,599196.768519,577500.0,108,99.000000,64.203704,1044.272397,8901.628985
868,2023,TAMPINES,576160.455487,565000.0,483,101.207039,68.672878,736.746165,13217.821338
869,2023,TOA PAYOH,593232.205128,510000.0,195,85.861538,63.256410,530.380552,6075.399559
870,2023,WOODLANDS,524988.255892,520000.0,594,102.033670,77.865320,602.072253,18376.758909


In [43]:
df_town_level_year.to_csv("resale_data_town_year.csv", index=False)

In [44]:
df_town_level_flat_type = df_kaggle.groupby(["town", "flat_type"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_remaining_lease=("years_remaining", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_town_level_flat_type

,town,flat_type,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_remaining_lease,avg_mrt_distance,avg_cbd_distance
0,ANG MO KIO,1 ROOM,48796.709957,50000.0,231,31.000000,80.735931,NaN,NaN
1,ANG MO KIO,2 ROOM,161293.787709,135500.0,716,44.438547,73.276536,422.530926,9743.555126
2,ANG MO KIO,3 ROOM,193907.302187,172000.0,32139,71.159339,75.541741,634.483742,9778.924449
3,ANG MO KIO,4 ROOM,309337.959729,270000.0,12391,93.237592,76.737632,602.293317,9724.376253
4,ANG MO KIO,5 ROOM,470325.334256,432250.0,5056,120.583861,79.036590,616.812802,9774.428540
...,...,...,...,...,...,...,...,...,...
132,YISHUN,4 ROOM,257685.965417,238000.0,32415,93.856640,83.166404,788.150323,15980.745021
133,YISHUN,5 ROOM,364064.509998,345000.0,9802,122.174046,84.370639,646.944178,15906.825216
134,YISHUN,EXECUTIVE,456147.900547,435000.0,3841,147.755793,83.969539,840.532918,15708.889103
135,YISHUN,MULTI GENERATION,428566.666667,455000.0,153,165.437908,90.869281,786.786917,15106.732906


In [45]:
df_town_level_flat_type.to_csv("resale_data_town_flat_type.csv", index=False)

In [46]:
df_town_level = df_kaggle.groupby(["town"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_remaining_lease=("years_remaining", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_town_level

,town,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_remaining_lease,avg_mrt_distance,avg_cbd_distance
0,ANG MO KIO,251483.095446,220000.0,50898,81.485206,76.198220,621.784869,9767.836171
1,BEDOK,270087.221066,240000.0,65157,88.166628,77.310174,757.167564,9997.803496
2,BISHAN,415156.147506,392000.0,20833,105.771468,83.127298,684.678778,7768.269660
3,BUKIT BATOK,268037.352559,243500.0,42583,92.931452,82.474790,634.387459,13670.205788
4,BUKIT MERAH,342340.615042,295000.0,33188,81.916205,76.019435,668.416113,3414.673438
5,BUKIT PANJANG,326040.797939,308000.0,26789,103.502744,85.955131,991.453424,14247.392300
6,BUKIT TIMAH,443494.732572,411000.0,2453,108.335915,77.565430,341.540015,8999.713360
7,CENTRAL AREA,333620.162127,255000.0,6939,72.962675,76.232454,300.818395,1762.174606
8,CHOA CHU KANG,340446.868494,332000.0,36964,111.900579,86.928606,678.344103,16392.987338
9,CLEMENTI,278522.461112,235000.0,27425,83.557557,76.611012,747.497735,10118.180433


In [47]:
df_town_level.to_csv("resale_data_town.csv", index=False)

## (B) Regional Level

In [ ]:
unique_towns = df_kaggle["town"].unique()
unique_towns

array(['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH',
       'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI',
       'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST',
       'KALLANG/WHAMPOA', 'MARINE PARADE', 'QUEENSTOWN', 'SENGKANG',
       'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN',
       'LIM CHU KANG', 'SEMBAWANG', 'BUKIT PANJANG', 'PASIR RIS',
       'PUNGGOL'], dtype=object)

In [55]:
region_map = {
    "NORTH": ["WOODLANDS", "YISHUN", "SEMBAWANG", "LIM CHU KANG"],
    "NORTH-EAST": ["ANG MO KIO", "HOUGANG", "SENGKANG", "PUNGGOL", "SERANGOON"],
    "EAST": ["BEDOK", "TAMPINES", "PASIR RIS"],
    "WEST": ["JURONG EAST", "JURONG WEST", "BUKIT BATOK", "CHOA CHU KANG", "BUKIT PANJANG", "CLEMENTI"],
    "CENTRAL": ["BISHAN", "KALLANG/WHAMPOA", "TOA PAYOH", "CENTRAL AREA", "BUKIT TIMAH",
                "GEYLANG", "MARINE PARADE", "QUEENSTOWN", "BUKIT MERAH"]
}

df_kaggle["region"] = df_kaggle["town"].map({town: region for region, towns in region_map.items() for town in towns})

In [58]:
df_region_level_year = df_kaggle.groupby(["year", "region"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_region_level_year

,year,region,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_mrt_distance,avg_cbd_distance
0,1990,CENTRAL,61607.181266,45000.0,3950,74.184051,613.531763,5473.417698
1,1990,EAST,76622.552783,65800.0,2084,91.490403,727.690556,11343.788667
2,1990,NORTH,59326.524807,52400.0,907,87.674752,734.091560,17496.958543
3,1990,NORTH-EAST,70223.733568,52330.0,2556,84.481221,710.082559,9587.548674
4,1990,WEST,71504.794548,56000.0,3008,89.199468,777.348251,13882.355104
...,...,...,...,...,...,...,...,...
165,2023,CENTRAL,627277.863929,613500.0,1242,85.527375,560.932137,5322.894424
166,2023,EAST,560510.180077,550000.0,1044,99.903257,795.993075,12306.327977
167,2023,NORTH,510852.266715,500000.0,1376,96.218023,713.395968,17406.796046
168,2023,NORTH-EAST,564778.250556,560000.0,1800,94.161667,1086.064624,12189.098554


In [59]:
df_region_level_year.to_csv("resale_data_region_year.csv", index=False)

In [60]:
df_region_level = df_kaggle.groupby(["region"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_region_level

,region,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_mrt_distance,avg_cbd_distance
0,CENTRAL,321174.245215,276000.0,183810,83.714145,610.717468,5571.234430
1,EAST,320357.047108,305000.0,175616,101.736300,784.450018,12323.801713
2,NORTH,291289.105886,278000.0,143909,100.192233,670.116894,17396.607076
3,NORTH-EAST,332145.915194,322000.0,170684,94.744991,894.151638,10886.944952
4,WEST,295077.477094,278000.0,222627,98.687742,778.185487,14842.548641


In [61]:
df_region_level.to_csv("resale_data_region.csv", index=False)